# REVE x Reference-Mismatch on BCIC-IV-2a

One controlled question: **a frozen REVE linear probe trained under one EEG
reference/spatial operator, how well does it transfer to the same subject and
task under a different operator in a later session?**

REVE's released IV-2a pipeline hardcodes CAR, so CAR is REVE's true operating
point and is used as the anchor throughout. We keep REVE's exact preprocessing
(CAR/operator -> 0.3-50 Hz -> 2-6 s -> 800 samples -> /100) and a strict frozen
encoder, and run a **within-subject, cross-session** protocol (train session ->
test session) to isolate the reference shift. This is deliberately *not* REVE's
cross-subject evaluation, so absolute numbers are not REVE's reported numbers.

The notebook runs five experiments:

1. **Mismatch matrix** (no adaptation and full-session EA): is there a gap, and does EA remove it?
2. **Depth trajectory**: where in the network does the gap appear?
3. **Trivial-shift control**: is the convention already obvious from raw input statistics?
4. **Alignment rescue**: is the task information still linearly present?
5. **Robustness**: does the gap survive pooling and PCA-dimension choices?

Start with `RUN_MODE = "smoke"` (subject 1). Switch to `"full"` once it passes.

## 0. Environment and REVE encoder

In [ ]:
import os, sys, json, time, shutil, hashlib, subprocess, warnings, logging, importlib
from pathlib import Path

WORK = Path('/kaggle/working')
REFSHIFT_DIR = WORK / 'Reference-Mismatch-MI-Net'
REVE_DIR = WORK / 'reve_eeg'
REVE_COMMIT = '06a7059a07c3dabd80aee60c3dbc1eca4bdbe1c7'

os.chdir(WORK)

# User project.
if not (REFSHIFT_DIR / 'refshift' / '__init__.py').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/JatinArutla/Reference-Mismatch-MI-Net.git', str(REFSHIFT_DIR)],
        check=True,
    )

# Official REVE repository, pinned to the commit used in the successful sanity runs.
if not (REVE_DIR / 'src' / 'dt.py').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/elouayas/reve_eeg.git', str(REVE_DIR)],
        check=True,
    )
subprocess.run(['git', '-C', str(REVE_DIR), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(REVE_DIR), 'checkout', REVE_COMMIT], check=True)

# Match the RefShift repository's own pinned MOABB version. Install both local
# packages without resolving their dependencies again, so pip cannot silently
# replace these explicit versions.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'moabb==1.5.0', 'mne==1.11.0', 'mne-bids>=0.18',
    'transformers==4.56.2', 'huggingface_hub', 'einops',
    'safetensors', 'scikit-learn==1.6.1', 'scipy', 'pandas',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-e', str(REFSHIFT_DIR), '--no-deps',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-e', str(REVE_DIR), '--no-deps',
], check=True)

# Editable installs write .pth metadata that a running notebook kernel may not
# notice until restart. Add both source trees explicitly so imports work now.
REFSHIFT_SRC = str(REFSHIFT_DIR)
REVE_SRC = str(REVE_DIR / 'src')
for source_path in (REFSHIFT_SRC, REVE_SRC):
    while source_path in sys.path:
        sys.path.remove(source_path)

# REVE src first, then RefShift project root.
sys.path.insert(0, REFSHIFT_SRC)
sys.path.insert(0, REVE_SRC)
importlib.invalidate_caches()

# Fail here, not several cells later, if either source tree is unavailable.
import refshift
from models.encoder import REVE as _REVE_IMPORT_CHECK

assert Path(refshift.__file__).resolve().is_relative_to(REFSHIFT_DIR.resolve()), (
    f'Imported refshift from the wrong location: {refshift.__file__}'
)
assert Path(sys.modules[_REVE_IMPORT_CHECK.__module__].__file__).resolve().is_relative_to(
    (REVE_DIR / 'src').resolve()
), 'The local pinned REVE source was not imported.'

# ---------------------------------------------------------------------
# Kaggle dataset wiring: use the RefShift repository's symlink helper.
# This must happen before any BNCI2014_001().get_data(...) call.
# ---------------------------------------------------------------------
from refshift import setup_kaggle_env

DEFAULT_IV2A_ROOT = Path(
    '/kaggle/input/datasets/delhialli/four-class-motor-imagery-bnci-001-2014'
)

if DEFAULT_IV2A_ROOT.exists():
    iv2a_source = DEFAULT_IV2A_ROOT
else:
    # Dataset slugs can change. Find a local attached copy, but never fall
    # back to a network download.
    candidates = sorted(Path('/kaggle/input').rglob('A01T.mat'))
    valid_roots = [p.parent for p in candidates if (p.parent / 'A01E.mat').exists()]
    valid_roots = list(dict.fromkeys(valid_roots))
    if len(valid_roots) != 1:
        raise FileNotFoundError(
            'Could not identify exactly one attached IV-2a directory containing '
            'A01T.mat and A01E.mat. Set REFSHIFT_IV2A_ROOT explicitly. '
            f'Candidates found: {valid_roots}'
        )
    iv2a_source = valid_roots[0]

os.environ['REFSHIFT_IV2A_ROOT'] = str(iv2a_source)
MNE_DATA_ROOT = WORK / 'mne_data'

setup_kaggle_env(
    mne_data=str(MNE_DATA_ROOT),
    moabb_results=str(WORK / 'moabb_results'),
    symlink_datasets=['iv2a'],
    thread_cap=1,
    verbose=True,
)

expected_cache = (
    MNE_DATA_ROOT
    / 'MNE-bnci-data'
    / '~bci'
    / 'database'
    / '001-2014'
)
required_iv2a = [
    expected_cache / 'A01T.mat',
    expected_cache / 'A01E.mat',
]

missing_iv2a = [p for p in required_iv2a if not (p.exists() or p.is_symlink())]
if missing_iv2a:
    raise FileNotFoundError(
        'IV-2a symlinks were not created; refusing to let MOABB download data. '
        f'Missing: {missing_iv2a}; source root: {iv2a_source}'
    )

print('IV-2a source:', iv2a_source)
print('MOABB cache:', expected_cache)
for p in required_iv2a:
    print(' ', p.name, '->', p.resolve())

# Version-safe Hugging Face authentication.
os.environ.pop('HF_TOKEN', None)
os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('Read Token')
except Exception:
    token = None

if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

import numpy as np
import pandas as pd
import torch
import mne
import sklearn
import scipy
import transformers
import moabb

mne.set_log_level('ERROR')
warnings.filterwarnings('ignore')
logging.getLogger('mne').setLevel(logging.ERROR)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'Enable a Kaggle GPU accelerator.'

refshift_commit = subprocess.run(
    ['git', '-C', str(REFSHIFT_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
reve_commit = subprocess.run(
    ['git', '-C', str(REVE_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()

print('device:', DEVICE)
print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('mne:', mne.__version__, '| moabb:', moabb.__version__, '| sklearn:', sklearn.__version__)
print('refshift module:', refshift.__file__)
print('refshift commit:', refshift_commit)
print('REVE commit:', reve_commit)

In [ ]:
from models.encoder import REVE
from downstream_tasks.position_utils import load_positions

REVE_ID = 'brain-bzh/reve-base'
reve, cls_query_token = REVE.from_pretrained(REVE_ID, cache_dir=str(WORK / '.cache'))
assert cls_query_token is not None, 'The pretrained cls_query_token was not found.'

reve = reve.eval().to(DEVICE)
cls_query_token = cls_query_token.detach().float().to(DEVICE)
for p in reve.parameters():
    p.requires_grad_(False)

print('embed_dim:', reve.embed_dim)
print('depth:', len(reve.transformer.layers))
print('patch_size:', reve.patch_size, '| overlap:', reve.overlap_size)
print('cls_query_token:', tuple(cls_query_token.shape), '| frozen:', not cls_query_token.requires_grad)

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, torch

RUN_MODE = "full"            # "smoke" (subject 1) or "full" (all 9)
SEED = 2026
np.random.seed(SEED); torch.manual_seed(SEED)

# REVE released IV-2a preprocessing. CAR is REVE's hardcoded operator; we vary it.
ELECTRODES = ["Fz","FC3","FC1","FCz","FC2","FC4","C5","C3","C1","Cz","C2","C4",
              "C6","CP3","CP1","CPz","CP2","CP4","P1","Pz","P2","POz"]
FS, BAND, FILTER_ORDER = 250, (0.3, 50.0), 5
WINDOW_S, N_OUT, SCALE_FACTOR = (2.0, 6.0), 800, 100.0

MODES = ("native", "car", "median", "rest", "cz_ref", "lap_small", "lap_large")
ANCHOR = "car"                # REVE's released reference
LAYERS = (0, 5, 11, 17, 22)   # for the depth trajectory

# Probe: scaler -> PCA -> logistic, C chosen by grouped CV on the train session.
PCA_COMPONENTS, C_GRID, CV_FOLDS = 128, (1e-4, 1e-3, 1e-2, 1e-1, 1.0), 5
RUN_BLOCK = 48               # IV-2a trials/run, used to form grouped-CV blocks

SUBJECTS = [1] if RUN_MODE == "smoke" else list(range(1, 10))
OUT = Path("/kaggle/working/reve_consolidated"); OUT.mkdir(parents=True, exist_ok=True)
print("subjects:", SUBJECTS, "| modes:", MODES, "| device:", DEVICE)

## 2. Preprocessing

REVE's released chain, with one fix: each session's runs are concatenated before
epoching so no trial is dropped at a run boundary. A fixed cue+6 s window is used;
because the band-pass is causal and we analyse [2,6] s, this matches REVE's
cue-to-next-cue result on the analysed window. The per-session sanity print below
reports the exact trial count.

In [ ]:
import mne
from scipy.signal import butter, lfilter, resample as scipy_resample
from moabb.datasets import BNCI2014_001
from refshift.references import apply_reference, build_graph, _ea_fit, _ea_apply
from downstream_tasks.position_utils import load_positions

LABEL_MAP = {"left_hand":0,"right_hand":1,"feet":2,"tongue":3,
             "769":0,"770":1,"771":2,"772":3}
_B, _A = butter(FILTER_ORDER, [BAND[0]/(0.5*FS), BAND[1]/(0.5*FS)], btype="band")
_norm = lambda d: str(d).strip().lower().replace(" ", "_")


def load_long_trials(subject):
    """Cue-aligned 0-6 s trials in raw microvolts, read session-continuous."""
    data = BNCI2014_001().get_data(subjects=[subject])
    required = int(WINDOW_S[1] * FS)
    X, y, sessions = [], [], []
    for session_name, runs in data[subject].items():
        raws = []
        for raw in runs.values():
            raw = raw.copy().pick_channels(ELECTRODES, ordered=True)
            if float(raw.info["sfreq"]) != FS:
                raw.resample(FS)
            raws.append(raw)
        raw = mne.concatenate_raws(raws)
        assert raw.ch_names == ELECTRODES
        raw_uV = raw.get_data(units="uV")
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        id_to_desc = {v: _norm(k) for k, v in event_id.items()}
        for sample, _, code_ in events:
            label = LABEL_MAP.get(id_to_desc.get(int(code_), ""))
            if label is None:
                continue
            seg = raw_uV[:, sample:sample + required]
            if seg.shape[-1] < required:        # only a session's final trial can be short
                continue
            X.append(seg); y.append(label); sessions.append(str(session_name))
    return np.stack(X).astype(np.float64), np.asarray(y, np.int64), np.asarray(sessions)


def reve_chain(X_long, mode, graph, car_after=False):
    """Operator -> (optional CAR) -> causal 0.3-50 Hz -> 2-6 s -> 800 samples -> /100."""
    X = apply_reference(X_long, mode, graph=graph)
    if car_after:
        X = apply_reference(X, "car", graph=graph)   # re-reference the operator output to CAR
    X = X.astype(np.float64)
    X = lfilter(_B, _A, X, axis=-1)
    X = X[..., int(WINDOW_S[0]*FS):int(WINDOW_S[1]*FS)]
    X = scipy_resample(X, N_OUT, axis=-1)
    return np.ascontiguousarray(X / SCALE_FACTOR, dtype=np.float32)


def session_split(sessions):
    train = sorted(set(map(str, sessions)))[0]   # "0train" sorts before "1test"
    tr = np.flatnonzero(sessions == train)
    te = np.flatnonzero(sessions != train)
    groups = np.arange(len(tr)) // RUN_BLOCK    # run-block groups for grouped CV
    return tr, te, groups


GRAPH = build_graph(ELECTRODES, include_rest=("rest" in MODES))
POS = load_positions(electrode_names=ELECTRODES).float().to(DEVICE)

# Sanity: confirm the session-continuous read recovers ~288 trials/session.
_X, _y, _s = load_long_trials(1)
for name in np.unique(_s):
    print(f"session {name}: {int((_s == name).sum())} trials")
print("per-trial shape:", _X.shape[1:], "| classes:", np.unique(_y))
del _X, _y, _s

## 3. Features and probe

Representation: REVE's released downstream embedding — the RMS-normed
concatenation of the attention-pooled query context and the token features
(`pooling="no"`), from a **frozen** encoder with the **frozen** pretrained
`cls_query_token`. This is a strict frozen-representation audit: we probe what is
linearly present in REVE's representation, we do not fine-tune. The probe is
scaler -> PCA -> logistic, with `C` chosen by group-stratified CV on the training
session (preprocessing refits inside each fold, so selection is properly nested).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score

def _rms(x, eps=1e-6):
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)

def _context(tokens):
    q = cls_query_token.expand(tokens.shape[0], -1, -1)
    s = torch.matmul(q, tokens.transpose(-1, -2)) / (reve.embed_dim ** 0.5)
    return torch.matmul(torch.softmax(s, dim=-1), tokens)        # (B, 1, D)

def _aggregate(tokens, pooling):
    if pooling == "context": return _rms(_context(tokens).squeeze(1))
    if pooling == "mean":    return _rms(tokens.mean(dim=1))
    if pooling == "no":      return _rms(torch.cat([_context(tokens), tokens], 1).flatten(1))
    raise ValueError(pooling)

@torch.inference_mode()
def reve_features(X, layer=None, pooling="no", batch=16):
    out = []
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i+batch]).float().to(DEVICE)
        pb = POS.unsqueeze(0).expand(xb.shape[0], -1, -1)
        tokens = reve(xb, pb, False) if layer is None else reve(xb, pb, True)[layer]
        out.append(_aggregate(tokens, pooling).float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def fit_probe(X, y, groups, *, pca=PCA_COMPONENTS, C=None):
    n_comp = min(pca, X.shape[1], len(X) - 1) if pca else None
    steps = [("scale", StandardScaler())]
    if n_comp:
        steps.append(("pca", PCA(n_components=n_comp, svd_solver="randomized", random_state=SEED)))
    steps.append(("clf", LogisticRegression(solver="lbfgs", max_iter=5000, random_state=SEED)))
    pipe = Pipeline(steps)
    if C is not None:
        return pipe.set_params(clf__C=C).fit(X, y)
    cv = StratifiedGroupKFold(n_splits=min(CV_FOLDS, len(np.unique(groups))),
                              shuffle=True, random_state=SEED)
    search = GridSearchCV(pipe, {"clf__C": list(C_GRID)}, scoring="balanced_accuracy",
                          cv=cv, refit=True)
    search.fit(X, y, groups=groups)
    return search.best_estimator_

def bacc(probe, X, y):
    return float(balanced_accuracy_score(y, probe.predict(X)))

## 4. Per-subject feature store

Extract context features once per subject for every operator, under no-adaptation
and full-session EA, plus the cheap input statistics. Experiments 1, 3 and 4 reuse
this store so REVE runs only once per subject.

In [ ]:
def ea_input(X_in, tr, te):
    Z = np.empty_like(X_in)
    Z[tr] = _ea_apply(X_in[tr], _ea_fit(X_in[tr]))
    Z[te] = _ea_apply(X_in[te], _ea_fit(X_in[te]))
    return Z

def build_store(subject):
    X_long, y, sessions = load_long_trials(subject)
    tr, te, groups = session_split(sessions)
    feat = {"no_ea": {}, "ea": {}, "car_after": {}}
    for m in MODES:
        X_in = reve_chain(X_long, m, GRAPH)
        feat["no_ea"][m]     = reve_features(X_in)
        feat["ea"][m]        = reve_features(ea_input(X_in, tr, te))
        feat["car_after"][m] = reve_features(reve_chain(X_long, m, GRAPH, car_after=True))
    return dict(subject=subject, y=y, tr=tr, te=te, groups=groups, feat=feat)

## Experiment 1 - Reference-mismatch transfer matrix

Train a probe on REVE features under each operator, test under every operator: the
diagonal is matched, the off-diagonal mismatched. A large matched-minus-mismatched
gap means the frozen readout is reference-fragile. We run it with no adaptation and
with independent full-session EA; if EA closes the gap, the shift is mostly
session-level covariance geometry rather than lost task information.

A third condition, **CAR-after-reference**, re-references each operator's output to CAR before encoding. Global references collapse to CAR exactly, so that block is degenerate by construction; the informative cells are global-to-spatial, reported separately.

In [ ]:
def experiment_matrix(store):
    y, tr, te, g = store["y"], store["tr"], store["te"], store["groups"]
    rows = []
    for cond in ("no_ea", "ea", "car_after"):
        f = store["feat"][cond]
        probes = {m: fit_probe(f[m][tr], y[tr], g) for m in MODES}
        for a in MODES:
            for b in MODES:
                rows.append(dict(subject=store["subject"], condition=cond,
                                 train_ref=a, test_ref=b,
                                 bacc=bacc(probes[a], f[b][te], y[te])))
    return rows

## 5. Run experiment 1 (one pass per subject)

In [ ]:
import gc
matrix_rows = []
for s in SUBJECTS:
    print(f"subject {s} ...", flush=True)
    store = build_store(s)
    matrix_rows += experiment_matrix(store)
    del store; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

matrix = pd.DataFrame(matrix_rows); matrix.to_csv(OUT/"matrix.csv", index=False)
print("saved matrix")

### Experiment 1 result

In [ ]:
def gap(df):
    diag = df[df.train_ref == df.test_ref].bacc.mean()
    off  = df[df.train_ref != df.test_ref].bacc.mean()
    return diag, off, diag - off

def matrix_of(cond):
    return (matrix[matrix.condition == cond]
            .groupby(["train_ref", "test_ref"]).bacc.mean()
            .unstack("test_ref").reindex(index=MODES, columns=MODES) * 100)

CONDS = ("no_ea", "ea", "car_after")
for cond in CONDS:
    d, o, gp = gap(matrix[matrix.condition == cond])
    print(f"{cond:9s} matched={100*d:5.1f}%  mismatched={100*o:5.1f}%  gap={100*gp:5.2f}pp")

for cond in CONDS:
    print(f"\n[{cond}] matrix (rows=train, cols=test), %:")
    print(matrix_of(cond).round(1).to_string())

# CAR-after collapses every global reference to CAR, so its global block is
# degenerate and its off-diagonal gap is deflated. The cells that test the
# hypothesis are global<->spatial: does re-CARing a Laplacian make it readable
# by a CAR-trained probe, or transfer a Laplacian probe to global references?
spatial = ["lap_small", "lap_large"]
glob = [m for m in MODES if m not in spatial]
print("\nGlobal<->spatial cross-family transfer, balanced accuracy %:")
for cond in CONDS:
    s = matrix[matrix.condition == cond]
    g2s = s[s.train_ref.isin(glob) & s.test_ref.isin(spatial)].bacc.mean() * 100
    s2g = s[s.train_ref.isin(spatial) & s.test_ref.isin(glob)].bacc.mean() * 100
    print(f"  {cond:9s} global->spatial={g2s:5.1f}  spatial->global={s2g:5.1f}")

# Subject-level transfer gap with a bootstrap CI over subjects (the paper number;
# 9 subjects, one deterministic probe each -> subject is the unit).
def gap_ci(df, n_boot=10000, seed=SEED):
    per = df.groupby("subject").apply(
        lambda d: d[d.train_ref == d.test_ref].bacc.mean()
                - d[d.train_ref != d.test_ref].bacc.mean())
    v = per.to_numpy(float)
    rng = np.random.default_rng(seed)
    boot = rng.choice(v, (n_boot, v.size), replace=True).mean(1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return v.mean() * 100, lo * 100, hi * 100, v.size

print("\nSubject-level gap [95% CI]:")
for cond in ("no_ea", "ea"):
    m, lo, hi, n = gap_ci(matrix[matrix.condition == cond])
    print(f"  {cond:7s} {m:5.2f}pp  [{lo:5.2f}, {hi:5.2f}]  n={n}")

## Operator invertibility (why global collapses and spatial does not)

The algebraic complement to the CAR-after result above. Global references are
contrast-preserving (H·M = H), so re-referencing collapses them to CAR. The
Laplacians are not, but their contrasts are still linearly recoverable and the
inversion is well-conditioned — the fixed-probe failure on them is a coordinate
change, not lost information (which is why target-covariance EA recovers it).

In [ ]:
from refshift import contrast_recovery_report
inv = contrast_recovery_report(ELECTRODES, modes=MODES)
print(inv.to_string(index=False))
inv.to_csv(OUT/"operator_invertibility.csv", index=False)

## Experiment 1b — Online few-trial target EA (the deployable test)

Experiment 1's EA whitens the whole test session at once, which you do not have at
deployment. Here a fixed CAR probe (trained on the full source session) meets a
target whitened from only the first `k` unlabeled test trials, evaluated on the
rest. `k="full"` uses the whole session as the endpoint, measured on the *same*
eval block so it is directly comparable to the `k` sweep. matched = test under
CAR, mismatched = mean over the other operators. Everything is CAR-anchored.

In [ ]:
K_GRID = (1, 2, 4, 8, 16, 32)
KMAX = max(K_GRID)

def whiten_prefix(target, k):
    # EA whitener fit on the first k target trials, applied to the held-out eval block.
    return _ea_apply(target[KMAX:], _ea_fit(target[:k]))

def whiten_full(target):
    # Full-session target EA on the same eval block: the k -> full endpoint,
    # measured identically so it is commensurable with the k sweep.
    return _ea_apply(target[KMAX:], _ea_fit(target))

kill_rows = []
for s in SUBJECTS:
    X_long, y, sessions = load_long_trials(s)
    tr, te, g = session_split(sessions)
    y_eval = y[te][KMAX:]
    car_in = reve_chain(X_long, ANCHOR, GRAPH)
    probe = fit_probe(reve_features(_ea_apply(car_in[tr], _ea_fit(car_in[tr]))), y[tr], g)
    for test_ref in MODES:
        target = reve_chain(X_long, test_ref, GRAPH)[te]
        for k in K_GRID:
            feat = reve_features(whiten_prefix(target, k))
            kill_rows.append(dict(subject=s, test_ref=test_ref, k=str(k),
                                  matched=(test_ref == ANCHOR), bacc=bacc(probe, feat, y_eval)))
        feat = reve_features(whiten_full(target))
        kill_rows.append(dict(subject=s, test_ref=test_ref, k="full",
                              matched=(test_ref == ANCHOR), bacc=bacc(probe, feat, y_eval)))
    del X_long; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

kill = pd.DataFrame(kill_rows); kill.to_csv(OUT/"online_ea.csv", index=False)
order = [str(k) for k in K_GRID] + ["full"]
piv = kill.groupby(["k", "matched"]).bacc.mean().unstack("matched") * 100
piv.columns = ["matched" if c else "mismatched" for c in piv.columns]
piv["gap"] = piv["matched"] - piv["mismatched"]
piv = piv.reindex(order)
print("Target EA by calibration trials k (CAR probe; k='full' = full-session target EA):")
print(piv.round(1).to_string())

## Experiment 2 - Depth trajectory

At each REVE layer, train the CAR probe and test matched (CAR) and mismatched
(every non-CAR operator). This shows where the gap forms: if matched task accuracy
rises with depth while mismatched transfer lags, the readout becomes more
operator-specific deeper in the network.

In [ ]:
traj_rows = []
for s in SUBJECTS:
    X_long, y, sessions = load_long_trials(s)
    tr, te, g = session_split(sessions)
    inputs = {m: reve_chain(X_long, m, GRAPH) for m in MODES}
    for layer in LAYERS:
        feat = {m: reve_features(inputs[m], layer=layer) for m in MODES}
        probe = fit_probe(feat[ANCHOR][tr], y[tr], g)
        matched = bacc(probe, feat[ANCHOR][te], y[te])
        mism = np.mean([bacc(probe, feat[m][te], y[te]) for m in MODES if m != ANCHOR])
        traj_rows.append(dict(subject=s, layer=layer, matched=matched, mismatched=mism))
    del inputs; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

traj = pd.DataFrame(traj_rows); traj.to_csv(OUT/"trajectory.csv", index=False)
t = traj.groupby("layer")[["matched","mismatched"]].mean()*100
t["gap"] = t.matched - t.mismatched
print("Mean over subjects by layer (%):"); print(t.round(1).to_string())

## 6. Outcome summary

In [ ]:
print("="*60)
print("REVE reference-mismatch: outcome summary")
print("="*60)
d0, o0, g0 = gap(matrix[matrix.condition == "no_ea"])
d1, o1, g1 = gap(matrix[matrix.condition == "ea"])
print(f"1.  Gap no-adapt : {100*g0:5.2f}pp (matched {100*d0:.1f} / mismatched {100*o0:.1f})")
print(f"    Gap full EA  : {100*g1:5.2f}pp  -> EA removes {100*(g0-g1):.1f}pp")

_sp = ["lap_small", "lap_large"]; _gl = [mm for mm in MODES if mm not in _sp]
def _xfam(cond):
    s = matrix[matrix.condition == cond]
    return (s[s.train_ref.isin(_gl) & s.test_ref.isin(_sp)].bacc.mean() * 100,
            s[s.train_ref.isin(_sp) & s.test_ref.isin(_gl)].bacc.mean() * 100)
_rw, _ca = _xfam("no_ea"), _xfam("car_after")
print(f"1c. CAR-after global<->spatial: g->s {_rw[0]:.1f}->{_ca[0]:.1f}  "
      f"s->g {_rw[1]:.1f}->{_ca[1]:.1f}  (global refs collapse to CAR; Laplacians do not)")

kg = kill.groupby("k").apply(
    lambda d: (d[d.matched].bacc.mean() - d[~d.matched].bacc.mean()) * 100)
print(f"1b. Online target EA gap: k=1 {kg.get('1', float('nan')):.1f}pp -> "
      f"k=32 {kg.get('32', float('nan')):.1f}pp -> full {kg.get('full', float('nan')):.1f}pp")

tg = traj.groupby("layer")[["matched", "mismatched"]].mean()
print(f"2.  Depth gap    : {100*(tg.matched-tg.mismatched).iloc[0]:.1f}pp (layer {LAYERS[0]}) -> "
      f"{100*(tg.matched-tg.mismatched).iloc[-1]:.1f}pp (layer {LAYERS[-1]})")